# Optimized Ensemble & Probabilistic ML for Freeze–Thaw Sandstone DCS
### Single-notebook implementation of the full study methodology

This notebook implements, end to end, the methodology developed for the
216-specimen freeze–thaw sandstone SHPB database: internal baseline
reproduction (MLR, **CSA-SVR**), an Optuna-tuned **10-learner pool**
(including a deep MLP), strict out-of-fold **stacking with exhaustive
enumeration of all 4,052 architectures** validated against **GA / ABC /
binary-CSA** searchers, a **probabilistic layer** (NGBoost Normal +
LogNormal, quantile-LightGBM, GPR, deep ensemble, **cross-conformal**
wrapper), repeated-split **significance testing** (Wilcoxon + Holm),
**SHAP / PDP-ICE interpretability**, and **cold-region design charts** with
5 %-fractile strengths and critical freeze–thaw life.

**How to use**
1. Set `DATA_PATH` and `MODE` in the configuration cell. `MODE="fast"` is a
   smoke test (minutes); **paper numbers require `MODE="full"` with seeds
   1–10** (hours — run overnight).
2. Run all cells top to bottom. **Interruptions are cheap**: heavy stages
   cache to disk under `OUT_DIR`, so after a crash or Colab disconnect just
   *Run All* again — completed tuning, the architecture-scan checkpoint and
   NGBoost tuning are reloaded, not recomputed. Caches are keyed to the
   configuration fingerprint (changing budgets invalidates them safely).

**Data & prior-art attribution (required in any publication):** data from
Meng et al. (2023) *Rock Mech Rock Eng* 56:2333–2355; ML benchmark
Zhou et al. (2024) *JRMGE* 16:2591–2606; also cite the SSA-RF study (IOP
Conf. Ser. EES 1337:012072, 2024 — confirm author list from the article
page) and Cai et al. (2025) *Materials* 18:3054, and differentiate as in the
methodology document.

In [ ]:
# Optional first-run installs (uncomment on Colab / a fresh machine):
# %pip install -q pandas numpy scipy scikit-learn openpyxl xgboost lightgbm catboost ngboost optuna shap matplotlib joblib
import hashlib
import json
import math
import os
import time
import warnings

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
try:                       # rich display in Jupyter, plain print elsewhere
    display
except NameError:
    display = print

import sklearn, xgboost, lightgbm, catboost, ngboost, optuna, shap
print("versions | sklearn", sklearn.__version__, "| xgb", xgboost.__version__,
      "| lgbm", lightgbm.__version__, "| cat", catboost.__version__,
      "| ngb", ngboost.__version__, "| optuna", optuna.__version__,
      "| shap", shap.__version__)

## Configuration
Everything the notebook does is deterministic given (`MODE`, seed). The
`fast` preset exercises identical code paths with small budgets; `full` is
the paper protocol (150 Optuna trials/learner, 10 seeds, 500-iteration
architecture searchers).

In [ ]:
# CONFIG-CELL ---------------------------------------------------------------
DATA_PATH = "FT_Sandstone_DCS_Dataset.xlsx"     # <- point at your file
for _cand in [DATA_PATH,
              "/mnt/user-data/uploads/FT_Sandstone_DCS_Dataset.xlsx",
              "ml_dataset_v1.csv"]:
    if os.path.exists(_cand):
        DATA_PATH = _cand
        break

OUT_DIR = "results_nb"
MODE = "fast"                     # "fast" = smoke test | "full" = paper run
SEEDS = [1] if MODE == "fast" else list(range(1, 11))
WALKTHROUGH_SEED = SEEDS[0]       # the seed narrated cell by cell below
RUN_BLOCKED = False               # condition-blocked stress split (Phase 2.2)

FEATURES = ["FTC_number", "ConfiningPressure_MPa", "ImpactPressure_MPa",
            "Porosity_pct", "CharImpedance_1e6_kg_s_m2"]
TARGET = "DCS_MPa"
ID_COL = "Specimen_ID"
CONDITION_COLS = ["FTC_number", "ConfiningPressure_MPa", "ImpactPressure_MPa"]
EXCEL_SHEET = "ML_Dataset"

TEST_SIZE, N_STRATA, KFOLD = 0.20, 10, 5
META_LEARNERS = ["ridge", "lasso", "linear", "gbdt_shallow"]
MIN_BASE_LEARNERS = 2
CONFORMAL_LEVELS = (0.90, 0.95)

BUDGETS = {
    "full": dict(optuna_trials=150, ngboost_trials=60, arch_pop=30,
                 arch_iters=500, csa_pop=30, csa_iters=200, mlp_max_iter=800,
                 tree_cap=1500, cat_cap=1500, gbdt_cap=1000, forest_cap=1000,
                 shap_nsamples=1024, shap_background=50, shap_explain=None,
                 pdp_grid=25, ice_lines=30, app_ftc_step=5, app_cp_step=0.5,
                 ngb_max_estimators=2000, deep_ensemble=5),
    "fast": dict(optuna_trials=10, ngboost_trials=6, arch_pop=16,
                 arch_iters=60, csa_pop=12, csa_iters=40, mlp_max_iter=250,
                 tree_cap=400, cat_cap=250, gbdt_cap=300, forest_cap=300,
                 shap_nsamples=200, shap_background=15, shap_explain=20,
                 pdp_grid=12, ice_lines=12, app_ftc_step=10, app_cp_step=1.0,
                 ngb_max_estimators=350, deep_ensemble=5),
}
B = BUDGETS[MODE]
os.makedirs(OUT_DIR, exist_ok=True)
print(f"data: {DATA_PATH}\nmode: {MODE} | seeds: {SEEDS} | out: {OUT_DIR}/")

In [ ]:
# GLUE-CELL: disk cache (resume-by-rerun) + in-memory results ledger ---------
def _fingerprint(*objs):
    js = json.dumps(objs, sort_keys=True, default=str)
    return hashlib.sha256(js.encode()).hexdigest()[:12]


CFG_FP = _fingerprint(MODE, B, KFOLD, TEST_SIZE, FEATURES, TARGET,
                      META_LEARNERS)


def seed_dir(seed, kind="random"):
    p = os.path.join(OUT_DIR, f"seed{seed}_{kind}")
    os.makedirs(p, exist_ok=True)
    return p


def cached(seed, kind, name, builder):
    """Build once, then reload on any later (re)run with the same config."""
    path = os.path.join(seed_dir(seed, kind), f"{name}_{CFG_FP}.joblib")
    if os.path.exists(path):
        print(f"  [cache] {name} (seed {seed}, {kind}) loaded")
        return joblib.load(path)
    obj = builder()
    joblib.dump(obj, path, compress=3)
    return obj


LEDGER_ROWS = []


def log_metrics(seed, kind, phase, model, metrics):
    """Idempotent logging: re-running a cell replaces, never duplicates."""
    global LEDGER_ROWS
    LEDGER_ROWS = [r for r in LEDGER_ROWS
                   if not (r["split_seed"] == seed and r["split_kind"] == kind
                           and r["model"] == model and r["metric"] in metrics)]
    for k, v in metrics.items():
        LEDGER_ROWS.append(dict(split_seed=seed, split_kind=kind, phase=phase,
                                model=model, metric=k, value=float(v)))


def ledger_frame():
    return pd.DataFrame(LEDGER_ROWS)


print("cache fingerprint:", CFG_FP)

## Core library (tested implementations)
The next seven cells define the study's machinery: data audit / splits /
metrics; the ten-learner pool with Optuna search spaces (paper Table T3);
the metaheuristics (continuous chameleon swarm for the SVR baseline; binary
GA / ABC / CSA for architecture search); out-of-fold stacking with the
checkpointed exhaustive enumerator; the uncertainty layer; and the
statistics / interpretability / application / figure functions. These are
verbatim the module implementations validated on this dataset.

In [ ]:
import hashlib
import os

import numpy as np
import pandas as pd
from scipy.stats import norm, spearmanr
from sklearn.model_selection import GroupShuffleSplit, train_test_split



# --------------------------------------------------------------------------- #
#  Loading and audit (Phase 1)
# --------------------------------------------------------------------------- #
def load_data(path):
    """Load the ML-ready table from .xlsx (sheet ML_Dataset) or .csv."""
    if str(path).lower().endswith((".xlsx", ".xlsm")):
        df = pd.read_excel(path, sheet_name=EXCEL_SHEET)
    else:
        df = pd.read_csv(path)
    missing = [c for c in FEATURES + [TARGET] if c not in df.columns]
    if missing:
        raise ValueError(f"Dataset is missing expected columns: {missing}")
    return df.reset_index(drop=True)


def audit(df, out_dir):
    """Structural audit + frozen CSV with SHA-256 (Phase 0 step 4 / Phase 1)."""
    report = {}
    report["n_rows"] = len(df)
    report["n_missing"] = int(df[FEATURES + [TARGET]].isna().sum().sum())
    grp = df.groupby(CONDITION_COLS)[TARGET]
    sizes = grp.size()
    report["n_conditions"] = int(len(sizes))
    report["all_conditions_have_3_replicates"] = bool((sizes == 3).all())
    cond = grp.agg(["mean", "median", "std"]).reset_index()
    cond.to_csv(os.path.join(out_dir, "condition_group_stats.csv"), index=False)

    frozen = os.path.join(out_dir, "ml_dataset_v1.csv")
    df.to_csv(frozen, index=False)
    with open(frozen, "rb") as fh:
        report["frozen_csv_sha256"] = hashlib.sha256(fh.read()).hexdigest()

    desc = df[FEATURES + [TARGET]].describe().T
    desc.to_csv(os.path.join(out_dir, "descriptive_statistics_T1.csv"))
    rho = df[FEATURES + [TARGET]].corr(method="spearman")
    rho.to_csv(os.path.join(out_dir, "spearman_matrix.csv"))
    return report, cond, rho


# --------------------------------------------------------------------------- #
#  Split protocol (Phase 2, steps 1-2)
# --------------------------------------------------------------------------- #
def make_split(df, seed, blocked=False):
    """
    Primary  : random 80/20 stratified by DCS deciles.
    Blocked  : whole FTC x CP x IP condition cells held out (stress test).
    Returns (train_df, test_df).
    """
    if blocked:
        groups = df[CONDITION_COLS].astype(str).agg("|".join, axis=1)
        gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=seed)
        tr_idx, te_idx = next(gss.split(df, groups=groups))
    else:
        strata = pd.qcut(df[TARGET], q=N_STRATA, labels=False, duplicates="drop")
        tr_idx, te_idx = train_test_split(
            np.arange(len(df)), test_size=TEST_SIZE,
            random_state=seed, stratify=strata,
        )
    return df.iloc[tr_idx].reset_index(drop=True), df.iloc[te_idx].reset_index(drop=True)


def xy(frame):
    return frame[FEATURES].to_numpy(float), frame[TARGET].to_numpy(float)


# --------------------------------------------------------------------------- #
#  Point metrics (Phase 2, step 5) — RMSE, MAE, R2, VAF: the baseline panel
# --------------------------------------------------------------------------- #
def rmse(y, p):
    return float(np.sqrt(np.mean((y - p) ** 2)))


def mae(y, p):
    return float(np.mean(np.abs(y - p)))


def r2(y, p):
    ss_res = np.sum((y - p) ** 2)
    ss_tot = np.sum((y - np.mean(y)) ** 2)
    return float(1.0 - ss_res / ss_tot)


def vaf(y, p):
    return float((1.0 - np.var(y - p) / np.var(y)) * 100.0)


def point_report(y, p):
    return dict(RMSE=rmse(y, p), MAE=mae(y, p), R2=r2(y, p), VAF=vaf(y, p))


# --------------------------------------------------------------------------- #
#  Uncertainty metrics (Phase 6) — NLL, CRPS, PICP, MPIW, reliability data
# --------------------------------------------------------------------------- #
def gaussian_nll(y, mu, sigma):
    sigma = np.clip(sigma, 1e-6, None)
    return float(-np.mean(norm.logpdf(y, loc=mu, scale=sigma)))


def gaussian_crps(y, mu, sigma):
    """Closed-form CRPS for a Normal predictive distribution."""
    sigma = np.clip(sigma, 1e-6, None)
    z = (y - mu) / sigma
    crps = sigma * (z * (2 * norm.cdf(z) - 1) + 2 * norm.pdf(z) - 1 / np.sqrt(np.pi))
    return float(np.mean(crps))


def interval_metrics(y, lo, hi, label=""):
    """PICP (coverage) and MPIW (mean width) for one interval level."""
    inside = (y >= lo) & (y <= hi)
    return {f"PICP{label}": float(np.mean(inside)),
            f"MPIW{label}": float(np.mean(hi - lo))}


def gaussian_intervals(mu, sigma, level):
    z = norm.ppf(0.5 + level / 2.0)
    return mu - z * sigma, mu + z * sigma


def reliability_curve(y, mu, sigma, levels=None):
    """Empirical coverage vs nominal level for a Gaussian predictive model."""
    if levels is None:
        levels = np.arange(0.50, 1.00, 0.05)
    emp = []
    for lv in levels:
        lo, hi = gaussian_intervals(mu, sigma, lv)
        emp.append(float(np.mean((y >= lo) & (y <= hi))))
    return np.asarray(levels), np.asarray(emp)


# --------------------------------------------------------------------------- #
#  Replicate-scatter validation data (Phase 6, step 5)
# --------------------------------------------------------------------------- #
def replicate_sigma_table(df_all, mu_all, sigma_all):
    """
    For every FTC x CP x IP condition group: observed SD of measured DCS vs the
    model's mean predicted sigma. Returned as a tidy frame + Spearman rho.
    """
    tmp = df_all[CONDITION_COLS].copy()
    tmp["obs"] = df_all[TARGET].to_numpy()
    tmp["sig"] = sigma_all
    g = tmp.groupby(CONDITION_COLS).agg(
        obs_sd=("obs", "std"), pred_sigma=("sig", "mean")).reset_index()
    rho, pval = spearmanr(g["obs_sd"], g["pred_sigma"])
    return g, float(rho), float(pval)

In [ ]:
import warnings

import numpy as np
import optuna
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import (ExtraTreesRegressor, GradientBoostingRegressor,
                              RandomForestRegressor)
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel, WhiteKernel
from sklearn.model_selection import KFold
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from xgboost import XGBRegressor


optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore")

POOL = ["svr", "knn", "rf", "et", "gbdt", "xgb", "lgbm", "cat", "gpr", "mlp"]


# --------------------------------------------------------------------------- #
#  Search spaces (Table T3) and estimator builders
# --------------------------------------------------------------------------- #
def suggest_params(name, trial, B):
    if name == "svr":
        return dict(C=trial.suggest_float("C", 0.01, 100, log=True),
                    gamma=trial.suggest_float("gamma", 0.01, 100, log=True),
                    epsilon=trial.suggest_float("epsilon", 0.001, 0.2))
    if name == "knn":
        return dict(n_neighbors=trial.suggest_int("n_neighbors", 2, 25),
                    weights=trial.suggest_categorical("weights", ["uniform", "distance"]),
                    p=trial.suggest_categorical("p", [1, 2]))
    if name in ("rf", "et"):
        return dict(n_estimators=trial.suggest_int("n_estimators", 100, B["forest_cap"]),
                    max_depth=trial.suggest_int("max_depth", 2, 20),
                    min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 10),
                    max_features=trial.suggest_float("max_features", 0.3, 1.0))
    if name == "gbdt":
        return dict(n_estimators=trial.suggest_int("n_estimators", 100, B["gbdt_cap"]),
                    learning_rate=trial.suggest_float("learning_rate", 0.005, 0.3, log=True),
                    max_depth=trial.suggest_int("max_depth", 2, 6),
                    subsample=trial.suggest_float("subsample", 0.5, 1.0))
    if name == "xgb":
        return dict(n_estimators=trial.suggest_int("n_estimators", 100, B["tree_cap"]),
                    learning_rate=trial.suggest_float("learning_rate", 0.005, 0.3, log=True),
                    max_depth=trial.suggest_int("max_depth", 2, 8),
                    subsample=trial.suggest_float("subsample", 0.5, 1.0),
                    colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0),
                    reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 10, log=True),
                    min_child_weight=trial.suggest_int("min_child_weight", 1, 10))
    if name == "lgbm":
        return dict(num_leaves=trial.suggest_int("num_leaves", 7, 63),
                    learning_rate=trial.suggest_float("learning_rate", 0.005, 0.3, log=True),
                    n_estimators=trial.suggest_int("n_estimators", 100, B["tree_cap"]),
                    feature_fraction=trial.suggest_float("feature_fraction", 0.5, 1.0),
                    bagging_fraction=trial.suggest_float("bagging_fraction", 0.5, 1.0),
                    bagging_freq=1,
                    reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 10, log=True))
    if name == "cat":
        return dict(depth=trial.suggest_int("depth", 3, 8),
                    learning_rate=trial.suggest_float("learning_rate", 0.005, 0.3, log=True),
                    iterations=trial.suggest_int("iterations",
                                                 min(200, B["cat_cap"]),
                                                 B["cat_cap"]),
                    l2_leaf_reg=trial.suggest_float("l2_leaf_reg", 1.0, 10.0))
    if name == "gpr":
        return dict(length_scale=trial.suggest_float("length_scale", 1e-2, 1e2, log=True),
                    alpha=trial.suggest_float("alpha", 1e-8, 1e-2, log=True),
                    const=trial.suggest_float("const", 0.1, 10, log=True))
    if name == "mlp":
        n_layers = trial.suggest_int("n_layers", 2, 5)
        width = trial.suggest_categorical("width", [32, 64, 128, 256])
        return dict(n_layers=n_layers, width=width,
                    lr=trial.suggest_float("lr", 1e-4, 1e-2, log=True),
                    alpha=trial.suggest_float("alpha", 1e-6, 1e-3, log=True),
                    batch=trial.suggest_categorical("batch", [16, 32, 64]))
    raise KeyError(name)


def build_estimator(name, params, seed, B):
    p = dict(params)
    if name == "svr":
        est = SVR(kernel="rbf", **p)
    elif name == "knn":
        est = KNeighborsRegressor(**p)
    elif name == "rf":
        est = RandomForestRegressor(random_state=seed, n_jobs=-1, **p)
    elif name == "et":
        est = ExtraTreesRegressor(random_state=seed, n_jobs=-1, **p)
    elif name == "gbdt":
        est = GradientBoostingRegressor(random_state=seed, **p)
    elif name == "xgb":
        est = XGBRegressor(random_state=seed, verbosity=0, tree_method="hist",
                           n_jobs=-1, **p)
    elif name == "lgbm":
        est = LGBMRegressor(random_state=seed, verbose=-1, n_jobs=-1, **p)
    elif name == "cat":
        est = CatBoostRegressor(random_seed=seed, logging_level="Silent",
                                allow_writing_files=False, **p)
    elif name == "gpr":
        kernel = (ConstantKernel(p["const"], (1e-3, 1e3))
                  * RBF(length_scale=np.full(len(FEATURES), p["length_scale"]),
                        length_scale_bounds=(1e-2, 1e2))
                  + WhiteKernel(noise_level=1e-2, noise_level_bounds=(1e-8, 1e1)))
        est = GaussianProcessRegressor(kernel=kernel, alpha=p["alpha"],
                                       normalize_y=True, random_state=seed,
                                       n_restarts_optimizer=1)
    elif name == "mlp":
        hidden = tuple([p["width"]] * p["n_layers"])
        est = MLPRegressor(hidden_layer_sizes=hidden, learning_rate_init=p["lr"],
                           alpha=p["alpha"], batch_size=p["batch"],
                           max_iter=B["mlp_max_iter"], early_stopping=True,
                           n_iter_no_change=25, random_state=seed, solver="adam")
    else:
        raise KeyError(name)
    return Pipeline([("scaler", StandardScaler()), ("model", est)])


# --------------------------------------------------------------------------- #
#  Per-learner tuning (Phase 4, step 1): Optuna TPE on 5-fold CV RMSE
# --------------------------------------------------------------------------- #
def cv_rmse(pipeline_factory, X, y, seed, k=KFOLD):
    kf = KFold(n_splits=k, shuffle=True, random_state=seed)
    errs = []
    for tr, va in kf.split(X):
        pipe = pipeline_factory()
        pipe.fit(X[tr], y[tr])
        errs.append(rmse(y[va], pipe.predict(X[va])))
    return float(np.mean(errs)), float(np.std(errs))


def tune_learner(name, X, y, seed, B):
    """Return (best_params, cv_mean, cv_std, best_so_far_history)."""
    history = []

    def objective(trial):
        params = suggest_params(name, trial, B)
        mean_err, _ = cv_rmse(lambda: build_estimator(name, params, seed, B),
                              X, y, seed)
        history.append(mean_err)
        return mean_err

    sampler = optuna.samplers.TPESampler(seed=seed)
    study = optuna.create_study(direction="minimize", sampler=sampler)
    study.optimize(objective, n_trials=B["optuna_trials"],
                   show_progress_bar=False, catch=(Exception,))
    best = study.best_params
    cv_mean, cv_std = cv_rmse(lambda: build_estimator(name, best, seed, B), X, y, seed)
    best_curve = np.minimum.accumulate(np.asarray(history))
    return best, cv_mean, cv_std, best_curve

In [ ]:
import numpy as np



# --------------------------------------------------------------------------- #
#  Continuous chameleon swarm (compact implementation)
# --------------------------------------------------------------------------- #
class ChameleonSwarm:
    def __init__(self, bounds, pop=30, iters=200, seed=0, p1=2.0, p2=2.0, pp=0.1):
        self.lo = np.array([b[0] for b in bounds], float)
        self.hi = np.array([b[1] for b in bounds], float)
        self.pop, self.iters, self.rng = pop, iters, np.random.default_rng(seed)
        self.p1, self.p2, self.pp = p1, p2, pp

    def minimize(self, func):
        d = len(self.lo)
        X = self.rng.uniform(self.lo, self.hi, size=(self.pop, d))
        V = np.zeros_like(X)
        F = np.array([func(x) for x in X])
        g = int(np.argmin(F))
        gbest, gfit = X[g].copy(), float(F[g])
        history = [gfit]
        for t in range(1, self.iters + 1):
            mu = 1.0 * np.exp(-(3.5 * t / self.iters) ** 3)   # shrinking local step
            pbest = X.copy()                                   # (memoryless variant)
            for i in range(self.pop):
                r1, r2, ri = self.rng.random(d), self.rng.random(d), self.rng.random()
                if ri >= self.pp:      # Eq. (4), exploration towards personal/global best
                    X[i] = X[i] + self.p1 * (pbest[i] - gbest) * r2 \
                                + self.p2 * (gbest - X[i]) * r1
                else:                  # Eq. (4), randomized local search
                    sgn = np.sign(self.rng.random(d) - 0.5)
                    X[i] = X[i] + mu * ((self.hi - self.lo) * self.rng.random(d)
                                        + self.lo) * sgn * 0.01
            # Eq. (5)-(6): tongue-attack (velocity) move towards the global best
            a = 2590.0 * (1.0 - np.exp(-np.log(max(t, 1))))
            Vnew = 0.5 * V + (gbest - X) * self.rng.random((self.pop, d))
            X = X + (Vnew ** 2 - V ** 2) / (2.0 * a + 1e-12)
            V = Vnew
            X = np.clip(X, self.lo, self.hi)
            F = np.array([func(x) for x in X])
            g = int(np.argmin(F))
            if F[g] < gfit:
                gbest, gfit = X[g].copy(), float(F[g])
            history.append(gfit)
        return gbest, gfit, np.asarray(history)


# --------------------------------------------------------------------------- #
#  Architecture encoding helpers
# --------------------------------------------------------------------------- #
N_BITS = 10                      # ten base learners in the pool (models.POOL)
N_META = len(META_LEARNERS)


def _valid(mask):
    return mask.sum() >= MIN_BASE_LEARNERS


def _random_candidate(rng):
    while True:
        mask = rng.integers(0, 2, N_BITS)
        if _valid(mask):
            return mask, int(rng.integers(0, N_META))


def _penalized(fitness, mask, meta):
    if not _valid(mask):
        return 1e9
    return fitness(tuple(mask.tolist()), meta)


class _History:
    def __init__(self):
        self.best = np.inf
        self.curve = []

    def push(self, f):
        self.best = min(self.best, f)
        self.curve.append(self.best)


# --------------------------------------------------------------------------- #
#  Binary genetic algorithm (Zhang et al., 2022 pedigree)
# --------------------------------------------------------------------------- #
def ga_search(fitness, pop=30, iters=200, seed=0, pmut=0.08):
    rng = np.random.default_rng(seed)
    P = [_random_candidate(rng) for _ in range(pop)]
    F = [_penalized(fitness, m, k) for m, k in P]
    hist = _History()
    for f in F:
        hist.push(f)
    for _ in range(iters):
        newP = []
        order = np.argsort(F)
        newP.append(P[order[0]])                       # elitism
        while len(newP) < pop:
            a, b = rng.choice(pop, 2, replace=False)   # tournament of 2, twice
            pa = P[a] if F[a] < F[b] else P[b]
            a, b = rng.choice(pop, 2, replace=False)
            pb = P[a] if F[a] < F[b] else P[b]
            cross = rng.random(N_BITS) < 0.5           # uniform crossover
            child_mask = np.where(cross, pa[0], pb[0]).astype(int)
            child_meta = pa[1] if rng.random() < 0.5 else pb[1]
            flip = rng.random(N_BITS) < pmut           # mutation
            child_mask = np.where(flip, 1 - child_mask, child_mask)
            if rng.random() < pmut:
                child_meta = int(rng.integers(0, N_META))
            newP.append((child_mask, child_meta))
        P = newP
        F = [_penalized(fitness, m, k) for m, k in P]
        for f in F:
            hist.push(f)
    i = int(np.argmin(F))
    return P[i], float(F[i]), np.asarray(hist.curve)


# --------------------------------------------------------------------------- #
#  Artificial bee colony (donor-faithful to Kardani et al., 2021)
# --------------------------------------------------------------------------- #
def abc_search(fitness, pop=30, iters=200, seed=0, limit=15):
    rng = np.random.default_rng(seed)
    P = [_random_candidate(rng) for _ in range(pop)]
    F = np.array([_penalized(fitness, m, k) for m, k in P], float)
    trials = np.zeros(pop, int)
    hist = _History()
    for f in F:
        hist.push(f)

    def neighbor(cand):
        mask, meta = cand[0].copy(), cand[1]
        if rng.random() < 0.7:                          # flip 1-2 inclusion bits
            for j in rng.choice(N_BITS, rng.integers(1, 3), replace=False):
                mask[j] = 1 - mask[j]
        else:                                           # or change the meta gene
            meta = int(rng.integers(0, N_META))
        return mask, meta

    for _ in range(iters):
        for i in range(pop):                            # employed bees
            cand = neighbor(P[i])
            f = _penalized(fitness, *cand)
            if f < F[i]:
                P[i], F[i], trials[i] = cand, f, 0
            else:
                trials[i] += 1
            hist.push(f)
        inv = 1.0 / (1.0 + F - F.min())                 # onlooker bees
        prob = inv / inv.sum()
        for _ in range(pop):
            i = int(rng.choice(pop, p=prob))
            cand = neighbor(P[i])
            f = _penalized(fitness, *cand)
            if f < F[i]:
                P[i], F[i], trials[i] = cand, f, 0
            else:
                trials[i] += 1
            hist.push(f)
        for i in range(pop):                            # scout bees
            if trials[i] > limit:
                P[i] = _random_candidate(rng)
                F[i] = _penalized(fitness, *P[i])
                trials[i] = 0
                hist.push(F[i])
    i = int(np.argmin(F))
    return P[i], float(F[i]), np.asarray(hist.curve)


# --------------------------------------------------------------------------- #
#  Binary chameleon swarm (sigmoid transfer of continuous positions)
# --------------------------------------------------------------------------- #
def bcsa_search(fitness, pop=30, iters=200, seed=0):
    rng = np.random.default_rng(seed)
    dim = N_BITS + 1
    X = rng.random((pop, dim))

    def decode(x):
        mask = (1.0 / (1.0 + np.exp(-8 * (x[:N_BITS] - 0.5))) > 0.5).astype(int)
        meta = int(min(N_META - 1, np.floor(x[N_BITS] * N_META)))
        return mask, meta

    F = np.array([_penalized(fitness, *decode(x)) for x in X], float)
    g = int(np.argmin(F))
    gbest, gfit = X[g].copy(), float(F[g])
    hist = _History()
    for f in F:
        hist.push(f)
    for t in range(1, iters + 1):
        mu = np.exp(-(3.5 * t / iters) ** 3)
        for i in range(pop):
            r1, r2, ri = rng.random(dim), rng.random(dim), rng.random()
            if ri >= 0.1:
                X[i] = X[i] + 2.0 * (gbest - X[i]) * r1 + 2.0 * (X[i] - gbest) * r2 * 0.1
            else:
                X[i] = X[i] + mu * (rng.random(dim) - 0.5) * 0.2
        X = np.clip(X, 0.0, 1.0)
        F = np.array([_penalized(fitness, *decode(x)) for x in X], float)
        g = int(np.argmin(F))
        if F[g] < gfit:
            gbest, gfit = X[g].copy(), float(F[g])
        for f in F:
            hist.push(f)
    return decode(gbest), gfit, np.asarray(hist.curve)

In [ ]:
import os
from itertools import combinations

import joblib

import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import LassoCV, LinearRegression, RidgeCV
from sklearn.model_selection import KFold



# --------------------------------------------------------------------------- #
#  Out-of-fold level-1 features (Phase 4, steps 2-3)
# --------------------------------------------------------------------------- #
def build_oof(X_tr, y_tr, X_te, tuned_params, seed, B, k=KFOLD):
    """
    Returns:
      oof      : (n_train, 10) matrix; entry [i, j] predicted by learner j
                 trained WITHOUT row i (strict out-of-fold).
      test_mat : (n_test, 10) predictions of full-train refits (for ensembling).
      fitted   : dict name -> pipeline fitted on the full training split.
    """
    n = len(X_tr)
    oof = np.full((n, len(POOL)), np.nan)
    kf = KFold(n_splits=k, shuffle=True, random_state=seed)
    fold_of = np.empty(n, int)
    for f, (tr, va) in enumerate(kf.split(X_tr)):
        fold_of[va] = f
        for j, name in enumerate(POOL):
            pipe = build_estimator(name, tuned_params[name], seed, B)
            pipe.fit(X_tr[tr], y_tr[tr])
            oof[va, j] = pipe.predict(X_tr[va])
    assert not np.isnan(oof).any(), "OOF matrix has holes — leakage guard failed"

    fitted, test_cols = {}, []
    for name in POOL:
        pipe = build_estimator(name, tuned_params[name], seed, B)
        pipe.fit(X_tr, y_tr)
        fitted[name] = pipe
        test_cols.append(pipe.predict(X_te))
    return oof, np.column_stack(test_cols), fitted, fold_of


# --------------------------------------------------------------------------- #
#  Meta-learners (ridge / lasso / linear / shallow GBDT)
# --------------------------------------------------------------------------- #
def make_meta(idx, seed):
    name = META_LEARNERS[idx]
    if name == "ridge":
        return RidgeCV(alphas=np.logspace(-3, 1, 9))
    if name == "lasso":
        return LassoCV(alphas=np.logspace(-3, 1, 9), max_iter=5000,
                       cv=3, random_state=seed)
    if name == "linear":
        return LinearRegression()
    if name == "gbdt_shallow":
        return GradientBoostingRegressor(n_estimators=100, max_depth=2,
                                         learning_rate=0.05, random_state=seed)
    raise KeyError(name)


# --------------------------------------------------------------------------- #
#  Memoized architecture fitness = 5-fold CV RMSE of meta on selected columns
# --------------------------------------------------------------------------- #
def fitness_factory(oof, y_tr, seed, k=KFOLD):
    memo = {}
    kf_idx = list(KFold(n_splits=k, shuffle=True, random_state=seed + 777)
                  .split(oof))

    def fitness(mask_tuple, meta_idx):
        key = (mask_tuple, meta_idx)
        if key in memo:
            return memo[key]
        cols = [j for j, b in enumerate(mask_tuple) if b]
        Z = oof[:, cols]
        errs = []
        for tr, va in kf_idx:
            meta = make_meta(meta_idx, seed)
            meta.fit(Z[tr], y_tr[tr])
            errs.append(rmse(y_tr[va], meta.predict(Z[va])))
        val = float(np.mean(errs))
        memo[key] = val
        return val

    return fitness, memo


# --------------------------------------------------------------------------- #
#  Exhaustive enumeration (Phase 5, step 4) — the search-quality ground truth
# --------------------------------------------------------------------------- #

def _arch_rmse(oof, y, folds, mask, meta_idx, seed):
    """Parallel worker: 5-fold CV RMSE of one architecture (deterministic)."""
    cols = [j for j, b in enumerate(mask) if b]
    Z = oof[:, cols]
    errs = []
    for tr, va in folds:
        meta = make_meta(meta_idx, seed)
        meta.fit(Z[tr], y[tr])
        errs.append(rmse(y[va], meta.predict(Z[va])))
    return float(np.mean(errs))


def exhaustive_search(oof, y_tr, seed, checkpoint=None, n_bits=10,
                      n_jobs=-1, chunk=250):
    """
    Enumerate ALL architectures (subset >= 2 of 10 learners x 4 metas = 4,052),
    in parallel, with an on-disk checkpoint so an interrupted run resumes.
    Fold indices use seed+777, identical to fitness_factory, so these values
    can seed the searcher memo exactly.
    """
    from joblib import Parallel, delayed
    folds = list(KFold(n_splits=KFOLD, shuffle=True,
                       random_state=seed + 777).split(oof))
    tasks = []
    for size in range(MIN_BASE_LEARNERS, n_bits + 1):
        for combo in combinations(range(n_bits), size):
            mask = tuple(1 if j in combo else 0 for j in range(n_bits))
            for m in range(len(META_LEARNERS)):
                tasks.append((mask, m))
    done = {}
    if checkpoint and os.path.exists(checkpoint):
        done = joblib.load(checkpoint)
        print(f"    exhaustive: resuming with {len(done)}/{len(tasks)} cached",
              flush=True)
    todo = [t for t in tasks if t not in done]
    for st in range(0, len(todo), chunk):
        batch = todo[st:st + chunk]
        vals = Parallel(n_jobs=n_jobs)(
            delayed(_arch_rmse)(oof, y_tr, folds, mk, mi, seed)
            for mk, mi in batch)
        done.update({t: v for t, v in zip(batch, vals)})
        if checkpoint:
            joblib.dump(done, checkpoint + ".tmp")
            os.replace(checkpoint + ".tmp", checkpoint)
        print(f"    exhaustive: {len(done)}/{len(tasks)} architectures scored",
              flush=True)
    df = pd.DataFrame([(mk, mi, done[(mk, mi)]) for mk, mi in tasks],
                      columns=["mask", "meta_idx", "cv_rmse"])
    best = df.loc[df["cv_rmse"].idxmin()]
    return df.sort_values("cv_rmse").reset_index(drop=True), best


# --------------------------------------------------------------------------- #
#  Metaheuristic searchers (Phase 5, step 3)
# --------------------------------------------------------------------------- #
def run_searchers(fitness, B, seed):
    out = {}
    for name, fn in [("GA", ga_search), ("ABC", abc_search), ("BCSA", bcsa_search)]:
        (mask, meta), f, curve = fn(fitness, pop=B["arch_pop"],
                                    iters=B["arch_iters"], seed=seed)
        out[name] = dict(mask=tuple(int(b) for b in np.atleast_1d(mask)),
                         meta_idx=int(meta), cv_rmse=float(f), curve=curve)
    return out


# --------------------------------------------------------------------------- #
#  Final leakage-safe fit and reference combiners (Phase 5, step 6)
# --------------------------------------------------------------------------- #
def fit_stack(mask, meta_idx, oof, y_tr, test_mat, seed):
    cols = [j for j, b in enumerate(mask) if b]
    meta = make_meta(meta_idx, seed)
    meta.fit(oof[:, cols], y_tr)            # meta sees only OOF features
    return meta.predict(test_mat[:, cols]), meta, cols


def reference_combiners(oof, y_tr, test_mat, seed):
    """Plain average of all learners + unoptimized full stack (all 10 + ridge)."""
    avg_pred = test_mat.mean(axis=1)
    full_mask = tuple([1] * len(POOL))
    full_pred, _, _ = fit_stack(full_mask, META_LEARNERS.index("ridge"),
                                oof, y_tr, test_mat, seed)
    return avg_pred, full_pred


def describe_arch(mask, meta_idx):
    names = [POOL[j] for j, b in enumerate(mask) if b]
    return "{" + "+".join(names) + "} -> " + META_LEARNERS[meta_idx]

In [ ]:
import math
import warnings

import numpy as np
import optuna
from lightgbm import LGBMRegressor
from ngboost import NGBRegressor
from ngboost.distns import LogNormal, Normal
from scipy.stats import lognorm, norm
from sklearn.model_selection import KFold
from sklearn.tree import DecisionTreeRegressor


optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore")


# --------------------------------------------------------------------------- #
#  NGBoost — tuned on 5-fold negative log-likelihood (Phase 6, step 1)
# --------------------------------------------------------------------------- #
def _ngb(params, dist, seed):
    return NGBRegressor(
        Dist=dist, n_estimators=params["n_estimators"],
        learning_rate=params["learning_rate"],
        minibatch_frac=params["minibatch_frac"],
        Base=DecisionTreeRegressor(max_depth=params["max_depth"],
                                   random_state=seed),
        random_state=seed, verbose=False)


def _dist_mu_sigma(model, X, dist):
    d = model.pred_dist(X)
    p = d.params
    if dist is Normal:
        return np.asarray(p["loc"]), np.asarray(p["scale"])
    # LogNormal: params s (shape) and scale; convert to mean/SD of the variable
    s, scale = np.asarray(p["s"]), np.asarray(p["scale"])
    mu = scale * np.exp(s ** 2 / 2.0)
    sd = mu * np.sqrt(np.exp(s ** 2) - 1.0)
    return mu, sd


def _dist_nll(model, X, y, dist):
    d = model.pred_dist(X)
    p = d.params
    if dist is Normal:
        return float(-np.mean(norm.logpdf(y, loc=p["loc"], scale=p["scale"])))
    return float(-np.mean(lognorm.logpdf(y, s=p["s"], scale=p["scale"])))


def tune_ngboost(X, y, seed, B, dist=Normal):
    def objective(trial):
        params = dict(
            n_estimators=trial.suggest_int("n_estimators", 200, B["ngb_max_estimators"]),
            learning_rate=trial.suggest_float("learning_rate", 0.005, 0.1, log=True),
            max_depth=trial.suggest_int("max_depth", 2, 5),
            minibatch_frac=trial.suggest_float("minibatch_frac", 0.5, 1.0))
        kf = KFold(n_splits=KFOLD, shuffle=True, random_state=seed)
        nlls = []
        for tr, va in kf.split(X):
            m = _ngb(params, dist, seed).fit(X[tr], y[tr])
            nlls.append(_dist_nll(m, X[va], y[va], dist))
        return float(np.mean(nlls))

    study = optuna.create_study(direction="minimize",
                                sampler=optuna.samplers.TPESampler(seed=seed))
    study.optimize(objective, n_trials=B["ngboost_trials"], show_progress_bar=False)
    return study.best_params


def eval_ngboost(X_tr, y_tr, X_te, y_te, params, seed, dist=Normal, tag="NGBoost"):
    m = _ngb(params, dist, seed).fit(X_tr, y_tr)
    mu, sd = _dist_mu_sigma(m, X_te, dist)
    res = point_report(y_te, mu)
    res["NLL"] = _dist_nll(m, X_te, y_te, dist)
    res["CRPS"] = gaussian_crps(y_te, mu, sd)     # Gaussian-approx CRPS for LogNormal
    for lv in CONFORMAL_LEVELS:
        lo, hi = gaussian_intervals(mu, sd, lv)
        res.update(interval_metrics(y_te, lo, hi, label=f"{int(lv * 100)}"))
    return m, mu, sd, res


# --------------------------------------------------------------------------- #
#  Quantile LightGBM (Phase 6, step 2a)
# --------------------------------------------------------------------------- #
def quantile_lgbm(X_tr, y_tr, X_te, y_te, lgbm_params, seed):
    def fit_q(a):
        p = {k: v for k, v in lgbm_params.items()}
        m = LGBMRegressor(objective="quantile", alpha=a, random_state=seed,
                          verbose=-1, n_jobs=-1, **p)
        m.fit(X_tr, y_tr)
        return m.predict(X_te)

    q = {a: fit_q(a) for a in (0.025, 0.05, 0.5, 0.95, 0.975)}
    res = point_report(y_te, q[0.5])
    res.update(interval_metrics(y_te, q[0.05], q[0.95], label="90"))
    res.update(interval_metrics(y_te, q[0.025], q[0.975], label="95"))
    return q, res


# --------------------------------------------------------------------------- #
#  GPR posterior intervals (Phase 6, step 2b) — reuse the tuned pool GPR
# --------------------------------------------------------------------------- #
def gpr_intervals(fitted_gpr_pipeline, X_te, y_te):
    scaler = fitted_gpr_pipeline.named_steps["scaler"]
    gp = fitted_gpr_pipeline.named_steps["model"]
    mu, sd = gp.predict(scaler.transform(X_te), return_std=True)
    res = point_report(y_te, mu)
    res["NLL"] = gaussian_nll(y_te, mu, sd)
    res["CRPS"] = gaussian_crps(y_te, mu, sd)
    for lv in CONFORMAL_LEVELS:
        lo, hi = gaussian_intervals(mu, sd, lv)
        res.update(interval_metrics(y_te, lo, hi, label=f"{int(lv * 100)}"))
    return mu, sd, res


# --------------------------------------------------------------------------- #
#  Deep ensemble of 5 MLPs (Phase 6, step 2c) — the DL uncertainty component
# --------------------------------------------------------------------------- #
def deep_ensemble(X_tr, y_tr, X_te, y_te, mlp_params, seed, B, n_members=5):
    preds = []
    for k in range(n_members):
        pipe = build_estimator("mlp", mlp_params, seed * 100 + k, B)
        pipe.fit(X_tr, y_tr)
        preds.append(pipe.predict(X_te))
    P = np.vstack(preds)
    mu, sd = P.mean(axis=0), P.std(axis=0) + 1e-6
    res = point_report(y_te, mu)
    res["NLL"] = gaussian_nll(y_te, mu, sd)
    res["CRPS"] = gaussian_crps(y_te, mu, sd)
    for lv in CONFORMAL_LEVELS:
        lo, hi = gaussian_intervals(mu, sd, lv)
        res.update(interval_metrics(y_te, lo, hi, label=f"{int(lv * 100)}"))
    return mu, sd, res


# --------------------------------------------------------------------------- #
#  Cross-conformal wrapper around the stack (Phase 6, step 3)
#  Nonconformity scores = |OOF residuals| of the stacked model; finite-sample
#  quantile correction ceil((n+1)(1-alpha))/n. Approximate validity (CV-style
#  cross-conformal; see Vovk 2005; Angelopoulos & Bates 2023).
# --------------------------------------------------------------------------- #
def conformal_from_oof(stack_oof_pred, y_tr, stack_test_pred, y_te,
                       levels=(0.90, 0.95)):
    scores = np.abs(y_tr - stack_oof_pred)
    n = len(scores)
    res, bands = {}, {}
    for lv in levels:
        rank = math.ceil((n + 1) * lv) / n
        qhat = float(np.quantile(scores, min(rank, 1.0)))
        lo, hi = stack_test_pred - qhat, stack_test_pred + qhat
        res.update(interval_metrics(y_te, lo, hi, label=f"{int(lv * 100)}"))
        bands[lv] = (lo, hi, qhat)
    return bands, res


def stack_oof_prediction(mask, meta_idx, oof, y_tr, seed):
    """OOF prediction of the *stack itself* (meta refit per fold on OOF cols),
    used both for conformal scores and as an honest training-side residual."""
    cols = [j for j, b in enumerate(mask) if b]
    Z = oof[:, cols]
    kf = KFold(n_splits=KFOLD, shuffle=True, random_state=seed + 777)
    pred = np.empty(len(y_tr))
    for tr, va in kf.split(Z):
        meta = make_meta(meta_idx, seed)
        meta.fit(Z[tr], y_tr[tr])
        pred[va] = meta.predict(Z[va])
    return pred

In [ ]:
import os
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats as st
from scipy.stats import norm


warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 120, "font.size": 9})


# =========================================================================== #
#  Phase 7 — repeated-split statistics
# =========================================================================== #
def paired_tests(ledger_df, champion, baselines, metric="RMSE",
                 kind="random", phase=("stack", "baseline", "pool")):
    """
    Wilcoxon signed-rank (primary) + paired t-test on per-split test metrics,
    Holm-corrected across the baseline family; Cohen's d (paired) effect size.
    """
    d = ledger_df[(ledger_df.metric == metric) & (ledger_df.split_kind == kind)
                  & (ledger_df.phase.isin(phase))]
    wide = d.pivot_table(index="split_seed", columns="model", values="value")
    rows = []
    for b in baselines:
        if champion not in wide or b not in wide:
            continue
        pair = wide[[champion, b]].dropna()
        x, yv = pair[champion].to_numpy(), pair[b].to_numpy()
        n = len(pair)
        row = dict(baseline=b, n_pairs=n,
                   champ_mean=float(x.mean()), champ_sd=float(x.std(ddof=1)) if n > 1 else np.nan,
                   base_mean=float(yv.mean()), base_sd=float(yv.std(ddof=1)) if n > 1 else np.nan)
        if n >= 2:
            diff = x - yv
            sd = diff.std(ddof=1)
            row["cohens_d"] = float(diff.mean() / sd) if sd > 0 else np.nan
            try:
                row["wilcoxon_p"] = float(st.wilcoxon(x, yv).pvalue)
            except ValueError:
                row["wilcoxon_p"] = np.nan
            row["ttest_p"] = float(st.ttest_rel(x, yv).pvalue)
            if n < 6:
                row["note"] = "n<6 pairs: p-values indicative only (run 10 seeds)"
        rows.append(row)
    out = pd.DataFrame(rows)
    if len(out) and "wilcoxon_p" in out:
        out["wilcoxon_p_holm"] = holm(out["wilcoxon_p"].to_numpy())
    return out


def holm(pvals):
    p = np.asarray(pvals, float)
    order = np.argsort(p)
    m = np.sum(~np.isnan(p))
    adj = np.full_like(p, np.nan)
    running = 0.0
    for rank, i in enumerate(order):
        if np.isnan(p[i]):
            continue
        running = max(running, (m - rank) * p[i])
        adj[i] = min(1.0, running)
    return adj


# =========================================================================== #
#  Phase 8 — interpretability
# =========================================================================== #
def shap_on_stack(stack_predict, X_tr, X_te, B, out_dir, tag="stack"):
    """KernelSHAP on the (model-agnostic) stacked predictor."""
    import shap
    bg = shap.kmeans(X_tr, min(B["shap_background"], len(X_tr)))
    n_expl = len(X_te) if B["shap_explain"] is None else min(B["shap_explain"], len(X_te))
    Xe = X_te[:n_expl]
    expl = shap.KernelExplainer(stack_predict, bg)
    sv = expl.shap_values(Xe, nsamples=B["shap_nsamples"])
    sv = np.asarray(sv)
    imp = np.abs(sv).mean(axis=0)
    pd.DataFrame({"feature": FEATURES, "mean_abs_SHAP": imp}) \
        .sort_values("mean_abs_SHAP", ascending=False) \
        .to_csv(os.path.join(out_dir, f"shap_importance_{tag}.csv"), index=False)

    fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
    order = np.argsort(imp)
    ax[0].barh([FEATURES[i] for i in order], imp[order], color="#2E5496")
    ax[0].set_xlabel("mean |SHAP| (MPa)")
    ax[0].set_title(f"F11a Global importance — {tag}")
    for j, f in enumerate(FEATURES):
        ax[1].scatter(sv[:, j], np.full(len(sv), j) + np.random.uniform(-.15, .15, len(sv)),
                      c=Xe[:, j], cmap="coolwarm", s=12)
    ax[1].set_yticks(range(len(FEATURES)))
    ax[1].set_yticklabels(FEATURES)
    ax[1].set_xlabel("SHAP value (MPa)")
    ax[1].set_title("F11b Beeswarm (colour = feature value)")
    fig.tight_layout()
    fig.savefig(os.path.join(out_dir, f"F11_shap_{tag}.png"))
    plt.show()
    return sv, imp


def xgb_interactions(fitted_xgb_pipeline, X_te, out_dir):
    """TreeSHAP interaction values on the tuned XGBoost (CP x IP coupling)."""
    import shap
    scaler = fitted_xgb_pipeline.named_steps["scaler"]
    booster = fitted_xgb_pipeline.named_steps["model"]
    Xs = scaler.transform(X_te)
    expl = shap.TreeExplainer(booster)
    inter = expl.shap_interaction_values(Xs)
    i_cp = FEATURES.index("ConfiningPressure_MPa")
    i_ip = FEATURES.index("ImpactPressure_MPa")
    i_ftc = FEATURES.index("FTC_number")
    fig, ax = plt.subplots(1, 2, figsize=(9, 3.4))
    ax[0].scatter(X_te[:, i_ip], inter[:, i_cp, i_ip] * 2, c=X_te[:, i_cp],
                  cmap="viridis", s=18)
    ax[0].set_xlabel("Impact pressure (MPa)")
    ax[0].set_ylabel("SHAP interaction CPxIP (MPa)")
    ax[0].set_title("F12a CP x IP coupling (colour = CP)")
    ax[1].scatter(X_te[:, i_ftc], inter[:, i_ftc, i_ftc], c=X_te[:, i_cp],
                  cmap="viridis", s=18)
    ax[1].set_xlabel("FTC number")
    ax[1].set_ylabel("SHAP main effect of FTC (MPa)")
    ax[1].set_title("F12b FTC effect (colour = CP)")
    fig.tight_layout()
    fig.savefig(os.path.join(out_dir, "F12_interactions_xgb.png"))
    plt.show()
    return inter


def pdp_ice(stack_predict, X_tr, B, out_dir,
            feats=("FTC_number", "ImpactPressure_MPa", "CharImpedance_1e6_kg_s_m2")):
    """Manual partial-dependence + ICE for the stack (Phase 8, step 4)."""
    rng = np.random.default_rng(0)
    idx_lines = rng.choice(len(X_tr), min(B["ice_lines"], len(X_tr)), replace=False)
    fig, axes = plt.subplots(1, len(feats), figsize=(3.4 * len(feats), 3.2))
    for ax, fname in zip(np.atleast_1d(axes), feats):
        j = FEATURES.index(fname)
        grid = np.linspace(X_tr[:, j].min(), X_tr[:, j].max(), B["pdp_grid"])
        ice = np.empty((len(idx_lines), len(grid)))
        for r, i in enumerate(idx_lines):
            Xrep = np.repeat(X_tr[i:i + 1], len(grid), axis=0)
            Xrep[:, j] = grid
            ice[r] = stack_predict(Xrep)
        Xall = np.repeat(X_tr, 1, axis=0)
        pdp = np.empty(len(grid))
        for gidx, gval in enumerate(grid):
            Xg = Xall.copy()
            Xg[:, j] = gval
            pdp[gidx] = stack_predict(Xg).mean()
        ax.plot(grid, ice.T, color="grey", alpha=0.25, lw=0.7)
        ax.plot(grid, pdp, color="#C00000", lw=2.2, label="PDP")
        ax.set_xlabel(fname)
        ax.set_ylabel("Predicted DCS (MPa)")
        ax.legend(fontsize=7)
    fig.suptitle("F13 PDP + ICE (stack)")
    fig.tight_layout()
    fig.savefig(os.path.join(out_dir, "F13_pdp_ice.png"))
    plt.show()


# =========================================================================== #
#  Phase 9 — engineering application module
# =========================================================================== #
def design_grid(df, B):
    ftc = np.arange(0, 100 + 1e-9, B["app_ftc_step"])
    cp = np.arange(0, 10 + 1e-9, B["app_cp_step"])
    ip = np.array([0.2, 0.4, 0.6])
    por = df["Porosity_pct"].mean()
    imp = df["CharImpedance_1e6_kg_s_m2"].mean()
    rows = [(f, c, i, por, imp) for i in ip for c in cp for f in ftc]
    G = pd.DataFrame(rows, columns=FEATURES)
    return G, ftc, cp, ip


def design_charts(ngb_model, dist_mu_sigma, df, cond_stats, B, out_dir,
                  threshold_frac=0.80):
    """
    Contours of mean and 5%-fractile DCS over FTC x CP at each IP; overlay the
    72 experimental group means; N_crit(CP, IP) at `threshold_frac` of FT0
    strength. Returns the fractile table (T12) and N_crit table.
    """
    G, ftc, cp, ip = design_grid(df, B)
    mu, sd = dist_mu_sigma(ngb_model, G[FEATURES].to_numpy(float))
    G["mu"], G["sd"] = mu, sd
    G["frac5"] = mu + norm.ppf(0.05) * sd

    fig, axes = plt.subplots(2, 3, figsize=(12, 6.4), sharex=True, sharey=True)
    for k, ipv in enumerate(ip):
        sub = G[np.isclose(G["ImpactPressure_MPa"], ipv)]
        MU = sub.pivot_table(index="ConfiningPressure_MPa",
                             columns="FTC_number", values="mu")
        F5 = sub.pivot_table(index="ConfiningPressure_MPa",
                             columns="FTC_number", values="frac5")
        for r, (Z, lab) in enumerate([(MU, "mean DCS"), (F5, "5% fractile DCS")]):
            ax = axes[r, k]
            cs = ax.contourf(Z.columns, Z.index, Z.values, levels=14, cmap="viridis")
            fig.colorbar(cs, ax=ax, shrink=0.85)
            ax.set_title(f"IP = {ipv} MPa — {lab}")
            if r == 1:
                ax.set_xlabel("FTC (cycles)")
            if k == 0:
                ax.set_ylabel("CP (MPa)")
        g = cond_stats[np.isclose(cond_stats["ImpactPressure_MPa"], ipv)]
        axes[0, k].scatter(g["FTC_number"], g["ConfiningPressure_MPa"],
                           c="white", s=14, edgecolor="k", lw=0.5,
                           label="experimental cells")
    axes[0, 0].legend(fontsize=7, loc="upper right")
    fig.suptitle("F16 Cold-region design charts (validation dots = 72 experimental cells)")
    fig.tight_layout()
    fig.savefig(os.path.join(out_dir, "F16_design_charts.png"))
    plt.show()

    # ---- N_crit curves --------------------------------------------------- #
    rows = []
    for ipv in ip:
        for cpv in cp:
            sub = G[np.isclose(G["ImpactPressure_MPa"], ipv)
                    & np.isclose(G["ConfiningPressure_MPa"], cpv)].sort_values("FTC_number")
            for col, tag in [("mu", "mean"), ("frac5", "frac5")]:
                ref = float(sub[col].iloc[0])       # FT0 strength for this class
                below = sub[sub[col] < threshold_frac * ref]
                ncrit = float(below["FTC_number"].iloc[0]) if len(below) else np.inf
                rows.append(dict(IP=ipv, CP=cpv, basis=tag, N_crit=ncrit))
    ncrit = pd.DataFrame(rows)
    ncrit.to_csv(os.path.join(out_dir, "N_crit_table.csv"), index=False)

    fig, ax = plt.subplots(figsize=(5.4, 3.4))
    for ipv, mk in zip(ip, ["o", "s", "^"]):
        d = ncrit[(ncrit.IP == ipv) & (ncrit.basis == "frac5")]
        ax.plot(d["CP"], d["N_crit"].replace(np.inf, 105), marker=mk, ms=3,
                label=f"IP={ipv} MPa (5% fractile)")
    ax.axhline(100, ls=":", c="grey")
    ax.set_xlabel("Confining pressure (MPa)")
    ax.set_ylabel(f"N_crit (cycles) at {int(threshold_frac * 100)}% of FT0 strength")
    ax.legend(fontsize=7)
    ax.set_title("F16b Critical freeze-thaw life")
    fig.tight_layout()
    fig.savefig(os.path.join(out_dir, "F16b_Ncrit.png"))
    plt.show()

    G.to_csv(os.path.join(out_dir, "design_grid_T12.csv"), index=False)
    return G, ncrit


# =========================================================================== #
#  Figure factory (subset used by the pipeline)
# =========================================================================== #
def fig_distributions(df, out_dir):
    cols = FEATURES + [TARGET]
    fig, axes = plt.subplots(1, len(cols), figsize=(2.1 * len(cols), 2.8))
    for ax, c in zip(axes, cols):
        ax.violinplot(df[c], showmedians=True)
        ax.boxplot(df[c], widths=0.12)
        ax.set_title(c, fontsize=7)
        ax.set_xticks([])
    fig.suptitle("F2 Distributions (n = 216)")
    fig.tight_layout()
    fig.savefig(os.path.join(out_dir, "F2_distributions.png"))
    plt.show()


def fig_corr(rho, out_dir):
    fig, ax = plt.subplots(figsize=(4.6, 3.8))
    im = ax.imshow(rho.values, cmap="coolwarm", vmin=-1, vmax=1)
    ax.set_xticks(range(len(rho))); ax.set_xticklabels(rho.columns, rotation=45,
                                                       ha="right", fontsize=7)
    ax.set_yticks(range(len(rho))); ax.set_yticklabels(rho.index, fontsize=7)
    for i in range(len(rho)):
        for j in range(len(rho)):
            ax.text(j, i, f"{rho.values[i, j]:.2f}", ha="center", va="center",
                    fontsize=6)
    fig.colorbar(im, shrink=0.8)
    ax.set_title("F3 Spearman correlations")
    fig.tight_layout()
    fig.savefig(os.path.join(out_dir, "F3_spearman.png"))
    plt.show()


def fig_parity(y_te, preds: dict, out_dir, tag="F6"):
    fig, axes = plt.subplots(1, len(preds), figsize=(3.4 * len(preds), 3.3))
    for ax, (name, p) in zip(np.atleast_1d(axes), preds.items()):
        ax.scatter(y_te, p, s=16, alpha=0.75, edgecolor="k", lw=0.3)
        lims = [min(y_te.min(), p.min()) - 5, max(y_te.max(), p.max()) + 5]
        ax.plot(lims, lims, "k--", lw=0.8)
        ax.set_xlim(lims); ax.set_ylim(lims)
        ax.set_xlabel("Actual DCS (MPa)"); ax.set_ylabel("Predicted DCS (MPa)")
        ax.set_title(f"{name}\nRMSE={rmse(y_te, p):.2f} MPa", fontsize=8)
    fig.suptitle(f"{tag} Predicted vs actual (test)")
    fig.tight_layout()
    fig.savefig(os.path.join(out_dir, f"{tag}_parity.png"))
    plt.show()


def fig_taylor(y_te, preds: dict, out_dir):
    """Compact Taylor diagram (F8): radius = SD ratio, angle = arccos(corr)."""
    ref_sd = np.std(y_te)
    fig = plt.figure(figsize=(4.6, 4.4))
    ax = fig.add_subplot(111, projection="polar")
    ax.set_thetamin(0); ax.set_thetamax(90)
    for name, p in preds.items():
        r = np.std(p) / ref_sd
        corr = np.corrcoef(y_te, p)[0, 1]
        ax.plot(np.arccos(np.clip(corr, -1, 1)), r, "o", ms=6, label=name)
    ax.plot(0, 1, "k*", ms=12, label="reference")
    ax.set_title("F8 Taylor diagram (test)", fontsize=9)
    ax.legend(fontsize=6, loc="upper right", bbox_to_anchor=(1.35, 1.1))
    fig.tight_layout()
    fig.savefig(os.path.join(out_dir, "F8_taylor.png"))
    plt.show()


def fig_arch_search(exh_df, searchers, out_dir):
    fig, ax = plt.subplots(1, 2, figsize=(9, 3.3))
    best = exh_df["cv_rmse"].min()
    for name, r in searchers.items():
        ax[0].plot(r["curve"], label=f"{name} (best {r['cv_rmse']:.3f})")
    ax[0].axhline(best, color="k", ls="--", lw=1,
                  label=f"exhaustive optimum {best:.3f}")
    ax[0].set_xlabel("fitness evaluations"); ax[0].set_ylabel("CV RMSE (MPa)")
    ax[0].legend(fontsize=6); ax[0].set_title("F5a Architecture search convergence")
    ax[1].hist(exh_df["cv_rmse"], bins=40, color="#2E5496")
    ax[1].axvline(best, color="k", ls="--")
    ax[1].set_xlabel("CV RMSE of architecture"); ax[1].set_ylabel("count")
    ax[1].set_title("F5b Landscape of all architectures")
    fig.tight_layout()
    fig.savefig(os.path.join(out_dir, "F5_architecture_search.png"))
    plt.show()


def fig_reliability(y_te, gaussians: dict, conformal_points, out_dir):
    fig, ax = plt.subplots(figsize=(4.4, 3.6))
    for name, (mu, sd) in gaussians.items():
        lv, emp = reliability_curve(y_te, mu, sd)
        ax.plot(lv, emp, marker="o", ms=3, label=name)
    for name, lv, emp in conformal_points:
        ax.plot(lv, emp, "k*", ms=9, label=name)
    ax.plot([0.5, 1.0], [0.5, 1.0], "k--", lw=0.8)
    ax.set_xlabel("Nominal coverage"); ax.set_ylabel("Empirical coverage")
    ax.set_title("F14a Reliability diagram")
    ax.legend(fontsize=6)
    fig.tight_layout()
    fig.savefig(os.path.join(out_dir, "F14a_reliability.png"))
    plt.show()


def fig_replicate_sigma(gtable, rho, out_dir):
    fig, ax = plt.subplots(figsize=(4.2, 3.4))
    ax.scatter(gtable["obs_sd"], gtable["pred_sigma"], s=18, alpha=0.8,
               edgecolor="k", lw=0.3)
    m = max(gtable["obs_sd"].max(), gtable["pred_sigma"].max()) * 1.05
    ax.plot([0, m], [0, m], "k--", lw=0.8)
    ax.set_xlabel("Observed within-condition SD (MPa)")
    ax.set_ylabel("Predicted sigma (MPa)")
    ax.set_title(f"F14b Replicate-scatter validation (Spearman rho = {rho:.2f})")
    fig.tight_layout()
    fig.savefig(os.path.join(out_dir, "F14b_replicate_sigma.png"))
    plt.show()


def fig_intervals(y_te, mu, bands90, out_dir, tag="NGBoost"):
    order = np.argsort(y_te)
    lo, hi = bands90
    x = np.arange(len(y_te))
    fig, ax = plt.subplots(figsize=(6.4, 3.4))
    inside = (y_te >= lo) & (y_te <= hi)
    ax.fill_between(x, lo[order], hi[order], alpha=0.3, label="90% PI")
    ax.plot(x, mu[order], lw=1, label="predicted mean")
    ax.scatter(x, y_te[order], c=np.where(inside[order], "#2E7D32", "#C62828"),
               s=14, label="actual (green=covered)")
    ax.set_xlabel("test samples (sorted by actual DCS)")
    ax.set_ylabel("DCS (MPa)")
    ax.set_title(f"F15 Prediction intervals — {tag}")
    ax.legend(fontsize=7)
    fig.tight_layout()
    fig.savefig(os.path.join(out_dir, f"F15_intervals_{tag}.png"))
    plt.show()


def fig_seed_boxplots(ledger_df, models, out_dir, metric="RMSE"):
    d = ledger_df[(ledger_df.metric == metric) & (ledger_df.split_kind == "random")]
    wide = d.pivot_table(index="split_seed", columns="model", values="value")
    cols = [m for m in models if m in wide.columns]
    if not cols or len(wide) < 2:
        return
    fig, ax = plt.subplots(figsize=(0.9 * len(cols) + 2, 3.2))
    ax.boxplot([wide[c].dropna() for c in cols], tick_labels=cols)
    ax.set_ylabel(f"test {metric} (MPa)")
    ax.set_title("F9 Repeated-split robustness")
    plt.xticks(rotation=30, ha="right", fontsize=7)
    fig.tight_layout()
    fig.savefig(os.path.join(out_dir, "F9_seed_boxplots.png"))
    plt.show()


def fig_residuals(y_te, pred, X_te, out_dir):
    res = y_te - pred
    i_ftc = FEATURES.index("FTC_number")
    fig, ax = plt.subplots(1, 2, figsize=(8.6, 3.2))
    ax[0].scatter(pred, res, s=16, alpha=0.8, edgecolor="k", lw=0.3)
    ax[0].axhline(0, color="k", lw=0.8, ls="--")
    ax[0].set_xlabel("Predicted DCS (MPa)"); ax[0].set_ylabel("Residual (MPa)")
    ax[0].set_title("F10a Residuals vs predicted")
    groups = sorted(np.unique(X_te[:, i_ftc]))
    ax[1].boxplot([res[X_te[:, i_ftc] == g] for g in groups],
                  tick_labels=[int(g) for g in groups])
    ax[1].axhline(0, color="k", lw=0.8, ls="--")
    ax[1].set_xlabel("FTC level"); ax[1].set_ylabel("Residual (MPa)")
    ax[1].set_title("F10b Residuals by FTC level")
    fig.tight_layout()
    fig.savefig(os.path.join(out_dir, "F10_residuals.png"))
    plt.show()

## Phase 0–1 — Load, audit, exploratory analysis
Structural audit (Phase 1), frozen CSV + SHA-256 (Phase 0.4), descriptive
statistics (paper **T1**), Spearman structure (**F3**) and distributions
(**F2**). The 72 condition-group statistics computed here anchor the
replicate-scatter uncertainty validation in Phase 6.

In [ ]:
df = load_data(DATA_PATH)
report, cond_stats, rho = audit(df, OUT_DIR)
print(report)
display(df[FEATURES + [TARGET]].describe().T.round(3))

In [ ]:
fig_distributions(df, OUT_DIR)     # F2
fig_corr(rho, OUT_DIR)             # F3

## Phase 2 — Fixed evaluation protocol
Random 80/20 split stratified by DCS deciles (repeated over `SEEDS`);
Z-scoring lives inside every Pipeline so it is refit per CV fold; the test
split is touched exactly once per model per seed. A condition-blocked stress
split is available via `RUN_BLOCKED=True`.

In [ ]:
tr_W, te_W = make_split(df, WALKTHROUGH_SEED)
Xtr_W, ytr_W = xy(tr_W)
Xte_W, yte_W = xy(te_W)
print(f"walkthrough seed {WALKTHROUGH_SEED}: train {len(tr_W)} / test {len(te_W)}"
      f" | test DCS range {yte_W.min():.1f}-{yte_W.max():.1f} MPa")

## Phase 3 — Internal baselines
MLR, the **reproduced CSA-SVR** (compact chameleon swarm over the published
hyperparameter ranges, identical evaluation budget discipline) and
default-parameter XGBoost. Reproducing the benchmark under *our* split
protocol is what makes every later comparison airtight.

In [ ]:
def phase_baselines(seed, kind, Xtr, ytr, Xte, yte, verbose=False):
    """Phase 3 — MLR, reproduced CSA-SVR (params cached), default XGBoost."""
    from sklearn.linear_model import LinearRegression
    mlr = Pipeline([("sc", StandardScaler()), ("m", LinearRegression())]).fit(Xtr, ytr)
    log_metrics(seed, kind, "baseline", "MLR", point_report(yte, mlr.predict(Xte)))

    def _tune_csa_svr():
        def obj(v):
            p = dict(C=10 ** v[0], gamma=10 ** v[1], epsilon=v[2])
            return cv_rmse(lambda: build_estimator("svr", p, seed, B),
                           Xtr, ytr, seed)[0]
        csa = ChameleonSwarm([(-2, 2), (-2, 2), (0.001, 0.2)],
                             pop=B["csa_pop"], iters=B["csa_iters"], seed=seed)
        v, f, _ = csa.minimize(obj)
        if verbose:
            print(f"  CSA-SVR reproduced: CV RMSE {f:.3f} | "
                  f"C={10**v[0]:.3g} gamma={10**v[1]:.3g} eps={v[2]:.3g}")
        return dict(C=10 ** v[0], gamma=10 ** v[1], epsilon=v[2])

    svr_params = cached(seed, kind, "csa_svr_params", _tune_csa_svr)
    svr = build_estimator("svr", svr_params, seed, B).fit(Xtr, ytr)
    svr_pred = svr.predict(Xte)
    log_metrics(seed, kind, "baseline", "CSA-SVR", point_report(yte, svr_pred))

    xd = Pipeline([("sc", StandardScaler()),
                   ("m", XGBRegressor(random_state=seed, verbosity=0,
                                      n_jobs=-1))]).fit(Xtr, ytr)
    log_metrics(seed, kind, "baseline", "XGB-default",
                point_report(yte, xd.predict(Xte)))
    return svr_pred


svr_pred_W = phase_baselines(WALKTHROUGH_SEED, "random",
                             Xtr_W, ytr_W, Xte_W, yte_W, verbose=True)
display(ledger_frame().query("phase=='baseline'")
        .pivot_table(index="model", columns="metric", values="value").round(4))

## Phase 4 — Ten-learner pool: individual optimization + out-of-fold matrix
Each learner gets the same Optuna-TPE budget on 5-fold CV RMSE (paper
**T3/T4/T5**). The OOF matrix (level-1 features) is built with a hard
leakage assert: every entry was predicted by models that never saw that row.
Both stages cache to disk — re-running is free.

In [ ]:
def phase_pool(seed, kind, Xtr, ytr, Xte, yte, verbose=False):
    """Phase 4 — tuned pool + strict OOF level-1 features (both cached)."""
    def _tune_all():
        tuned, rows = {}, []
        for name in POOL:
            t0 = time.time()
            best, cvm, cvs, _ = tune_learner(name, Xtr, ytr, seed, B)
            tuned[name] = best
            rows.append(dict(learner=name, cv_rmse_mean=cvm, cv_rmse_std=cvs,
                             seconds=round(time.time() - t0, 1)))
            if verbose:
                print(f"  tuned {name:5s}: CV RMSE {cvm:.3f} +/- {cvs:.3f} "
                      f"({time.time() - t0:.0f}s)")
        return dict(tuned=tuned, cv_table=pd.DataFrame(rows))

    pk = cached(seed, kind, "tuned_pool", _tune_all)

    def _oof():
        oof, test_mat, fitted, fold_of = build_oof(Xtr, ytr, Xte,
                                                   pk["tuned"], seed, B)
        return dict(oof=oof, test_mat=test_mat, fitted=fitted, fold_of=fold_of)

    op = cached(seed, kind, "oof_pack", _oof)
    for j, name in enumerate(POOL):
        log_metrics(seed, kind, "pool", f"single:{name}",
                    point_report(yte, op["test_mat"][:, j]))
    best_j = int(np.argmin([rmse(yte, op["test_mat"][:, j])
                            for j in range(len(POOL))]))
    log_metrics(seed, kind, "pool", "BestSingle",
                point_report(yte, op["test_mat"][:, best_j]))
    if verbose:
        print(f"  best single on test: {POOL[best_j]} "
              f"(RMSE {rmse(yte, op['test_mat'][:, best_j]):.3f})")
    return (pk["tuned"], pk["cv_table"], op["oof"], op["test_mat"],
            op["fitted"], POOL[best_j])


(tuned_W, cv_table_W, oof_W, test_mat_W,
 fitted_W, best_single_W) = phase_pool(WALKTHROUGH_SEED, "random",
                                       Xtr_W, ytr_W, Xte_W, yte_W, verbose=True)
pd.DataFrame(tuned_W).to_json(os.path.join(seed_dir(WALKTHROUGH_SEED),
                                           "T4_tuned_params.json"))
cv_table_W.to_csv(os.path.join(seed_dir(WALKTHROUGH_SEED), "T5_pool_cv.csv"),
                  index=False)
display(cv_table_W.round(3))

## Phase 5 — Architecture search: exhaustive ground truth + three searchers
The candidate space (subsets ≥ 2 of the 10 learners × 4 meta-learners =
4,052 architectures) is **fully enumerated** — checkpointed to disk every
250 evaluations and keyed to a hash of the OOF matrix, so interruptions
resume and stale caches are impossible. GA (Zhang et al. 2022 pedigree),
ABC (Kardani et al. 2021) and binary-CSA then replay against the known
global optimum: the search-quality validation few published stacking papers
provide (**T6, F5**).

In [ ]:
def phase_stack(seed, kind, oof, ytr, test_mat, yte, verbose=False):
    """Phase 5 — exhaustive + GA/ABC/BCSA + leakage-safe final stack."""
    fitness, memo = fitness_factory(oof, ytr, seed)
    oof_fp = hashlib.sha256(oof.tobytes()).hexdigest()[:10]
    ck = os.path.join(seed_dir(seed, kind), f"exh_{CFG_FP}_{oof_fp}.joblib")
    exh_df, exh_best = exhaustive_search(oof, ytr, seed, checkpoint=ck)
    memo.update({(tuple(t.mask), int(t.meta_idx)): float(t.cv_rmse)
                 for t in exh_df.itertuples()})
    searchers = run_searchers(fitness, B, seed)
    mask, meta_idx = tuple(exh_best["mask"]), int(exh_best["meta_idx"])
    agree = {k: bool(v["cv_rmse"] <= exh_best["cv_rmse"] + 1e-9)
             for k, v in searchers.items()}
    stack_pred, meta, cols = fit_stack(mask, meta_idx, oof, ytr, test_mat, seed)
    avg_pred, full_pred = reference_combiners(oof, ytr, test_mat, seed)
    log_metrics(seed, kind, "stack", "AvgEnsemble", point_report(yte, avg_pred))
    log_metrics(seed, kind, "stack", "FullStack", point_report(yte, full_pred))
    log_metrics(seed, kind, "stack", "SelectedStack",
                point_report(yte, stack_pred))
    exh_df.to_csv(os.path.join(seed_dir(seed, kind),
                               "T6_exhaustive_architectures.csv"), index=False)
    with open(os.path.join(seed_dir(seed, kind),
                           "selected_architecture.json"), "w") as fh:
        json.dump(dict(description=describe_arch(mask, meta_idx),
                       cv_rmse=float(exh_best["cv_rmse"]),
                       searcher_agreement=agree), fh, indent=2)
    if verbose:
        print(f"  exhaustive optimum: {describe_arch(mask, meta_idx)} "
              f"(CV RMSE {exh_best['cv_rmse']:.3f})")
        print(f"  searchers reached the global optimum: {agree}")
    return dict(mask=mask, meta_idx=meta_idx, meta=meta, cols=cols,
                stack_pred=stack_pred, avg_pred=avg_pred,
                full_pred=full_pred, exh_df=exh_df, searchers=searchers,
                agree=agree)


sk_W = phase_stack(WALKTHROUGH_SEED, "random", oof_W, ytr_W,
                   test_mat_W, yte_W, verbose=True)
fig_arch_search(sk_W["exh_df"], sk_W["searchers"],
                seed_dir(WALKTHROUGH_SEED))            # F5

In [ ]:
# Walkthrough-seed comparison table (one row block of paper T7) + headline figs
rows = [("MLR / CSA-SVR / XGB-default: see Phase 3 table", None)]
tbl = pd.DataFrame([
    dict(model="Reproduced CSA-SVR", **point_report(yte_W, svr_pred_W)),
    dict(model=f"Best single ({best_single_W})",
         **point_report(yte_W, test_mat_W[:, POOL.index(best_single_W)])),
    dict(model="Average ensemble", **point_report(yte_W, sk_W["avg_pred"])),
    dict(model="Full stack (10 + ridge)",
         **point_report(yte_W, sk_W["full_pred"])),
    dict(model="Selected stack", **point_report(yte_W, sk_W["stack_pred"])),
]).set_index("model")
display(tbl.round(4))

fig_parity(yte_W, {"SelectedStack": sk_W["stack_pred"],
                   "CSA-SVR": svr_pred_W,
                   f"best single ({best_single_W})":
                       test_mat_W[:, POOL.index(best_single_W)]},
           seed_dir(WALKTHROUGH_SEED))                 # F6
fig_taylor(yte_W, {"SelectedStack": sk_W["stack_pred"],
                   "CSA-SVR": svr_pred_W,
                   "FullStack": sk_W["full_pred"],
                   "AvgEnsemble": sk_W["avg_pred"]},
           seed_dir(WALKTHROUGH_SEED))                 # F8
fig_residuals(yte_W, sk_W["stack_pred"], Xte_W,
              seed_dir(WALKTHROUGH_SEED))              # F10

## Phase 6 — Probabilistic layer + conformal wrapper
NGBoost is tuned on 5-fold **NLL** (not RMSE) with a LogNormal ablation;
benchmarks are quantile-LightGBM, GPR posterior, and a 5-member deep MLP
ensemble. The **cross-conformal** wrapper converts the selected stack's OOF
residuals into distribution-free intervals with a finite-sample quantile
correction. Calibration (**F14a**), intervals (**F15**) and — unique to this
dataset — the **replicate-scatter validation** of predicted sigma against the
observed within-condition SD of the 72 experimental groups (**F14b**).

In [ ]:
def phase_uq(seed, kind, Xtr, ytr, Xte, yte, tuned, fitted, oof, sk,
             verbose=False):
    """Phase 6 — NGBoost (+LogN), qLGBM, GPR, deep ensemble, conformal."""
    ngb_params = cached(seed, kind, "ngb_params",
                        lambda: tune_ngboost(Xtr, ytr, seed, B, dist=Normal))
    ngb, mu_n, sd_n, res_n = eval_ngboost(Xtr, ytr, Xte, yte, ngb_params,
                                          seed, dist=Normal)
    log_metrics(seed, kind, "uq", "NGBoost", res_n)
    _, mu_l, sd_l, res_l = eval_ngboost(Xtr, ytr, Xte, yte, ngb_params,
                                        seed, dist=LogNormal)
    log_metrics(seed, kind, "uq", "NGBoost-LogN", res_l)
    _, res_q = quantile_lgbm(Xtr, ytr, Xte, yte, tuned["lgbm"], seed)
    log_metrics(seed, kind, "uq", "qLGBM", res_q)
    mu_g, sd_g, res_g = gpr_intervals(fitted["gpr"], Xte, yte)
    log_metrics(seed, kind, "uq", "GPR-UQ", res_g)
    mu_d, sd_d, res_d = deep_ensemble(Xtr, ytr, Xte, yte, tuned["mlp"],
                                      seed, B, n_members=B["deep_ensemble"])
    log_metrics(seed, kind, "uq", "DeepEnsemble", res_d)
    soof = stack_oof_prediction(sk["mask"], sk["meta_idx"], oof, ytr, seed)
    bands, res_c = conformal_from_oof(soof, ytr, sk["stack_pred"], yte,
                                      levels=CONFORMAL_LEVELS)
    res_c.update(point_report(yte, sk["stack_pred"]))
    log_metrics(seed, kind, "uq", "Conformal(stack)", res_c)
    if verbose:
        print("  UQ logged: NGBoost, NGBoost-LogN, qLGBM, GPR-UQ, "
              "DeepEnsemble, Conformal(stack)")
    return dict(ngb=ngb, mu_n=mu_n, sd_n=sd_n, mu_g=mu_g, sd_g=sd_g,
                mu_d=mu_d, sd_d=sd_d, bands=bands)


uq_W = phase_uq(WALKTHROUGH_SEED, "random", Xtr_W, ytr_W, Xte_W, yte_W,
                tuned_W, fitted_W, oof_W, sk_W, verbose=True)
uq_view = (ledger_frame()
           .query("split_seed==@WALKTHROUGH_SEED and split_kind=='random' "
                  "and phase=='uq'")
           .pivot_table(index="model", columns="metric", values="value"))
keep = [c for c in ["RMSE", "MAE", "R2", "NLL", "CRPS",
                    "PICP90", "MPIW90", "PICP95", "MPIW95"] if c in uq_view]
display(uq_view[keep].round(4))

In [ ]:
# Calibration + interval figures + replicate-scatter validation
conf_pts = [("Conformal(stack)", lv,
             float(np.mean((yte_W >= uq_W["bands"][lv][0])
                           & (yte_W <= uq_W["bands"][lv][1]))))
            for lv in CONFORMAL_LEVELS]
fig_reliability(yte_W, {"NGBoost": (uq_W["mu_n"], uq_W["sd_n"]),
                        "GPR": (uq_W["mu_g"], uq_W["sd_g"]),
                        "DeepEns": (uq_W["mu_d"], uq_W["sd_d"])},
                conf_pts, seed_dir(WALKTHROUGH_SEED))          # F14a
lo90, hi90 = gaussian_intervals(uq_W["mu_n"], uq_W["sd_n"], 0.90)
fig_intervals(yte_W, uq_W["mu_n"], (lo90, hi90),
              seed_dir(WALKTHROUGH_SEED), tag="NGBoost")       # F15
fig_intervals(yte_W, sk_W["stack_pred"], uq_W["bands"][0.90][:2],
              seed_dir(WALKTHROUGH_SEED), tag="Conformal_stack")

# Predicted sigma vs observed within-condition SD (72 groups).
# NOTE: sigma is predicted for all 216 specimens with the train-fitted model,
# so train rows are included by construction — state this caveat (see
# methodology Phase 6.5) or restrict to test-only conditions as a check.
mu_all, sd_all = _dist_mu_sigma(uq_W["ngb"], df[FEATURES].to_numpy(float),
                                Normal)
gtab, rho_s, pval_s = replicate_sigma_table(df, mu_all, sd_all)
gtab.to_csv(os.path.join(seed_dir(WALKTHROUGH_SEED),
                         "replicate_sigma_validation.csv"), index=False)
fig_replicate_sigma(gtab, rho_s, seed_dir(WALKTHROUGH_SEED))   # F14b
print(f"replicate-scatter validation: Spearman rho = {rho_s:.2f} "
      f"(p = {pval_s:.3f})")

## Phase 7 — Repeated splits + statistical significance
`run_one_seed` composes Phases 3–6 for any seed/split-kind; the loop below
runs every seed in `SEEDS` (the walkthrough seed is already logged — its
caches make a re-call free). Then: paper **T7/T8** aggregates, Wilcoxon +
paired-t with **Holm** correction and Cohen's d, and the **F9** robustness
boxplots. With fewer than ~6 seeds the p-values are indicative only — the
paper protocol is all ten.

In [ ]:
def run_one_seed(seed, kind="random", verbose=False):
    tr, te = make_split(df, seed, blocked=(kind == "blocked"))
    Xtr, ytr = xy(tr)
    Xte, yte = xy(te)
    print(f"=== seed {seed} [{kind}] | train {len(tr)} / test {len(te)} ===")
    t0 = time.time()
    phase_baselines(seed, kind, Xtr, ytr, Xte, yte, verbose)
    tuned, _, oof, test_mat, fitted, _ = phase_pool(seed, kind, Xtr, ytr,
                                                    Xte, yte, verbose)
    sk = phase_stack(seed, kind, oof, ytr, test_mat, yte, verbose)
    phase_uq(seed, kind, Xtr, ytr, Xte, yte, tuned, fitted, oof, sk, verbose)
    print(f"=== seed {seed} [{kind}] done in {(time.time()-t0)/60:.1f} min ===")


for _seed in SEEDS:
    if _seed == WALKTHROUGH_SEED:
        continue                       # already logged by the walkthrough
    run_one_seed(_seed)

if RUN_BLOCKED:
    for _seed in SEEDS:
        run_one_seed(_seed, kind="blocked")

In [ ]:
led = ledger_frame()
led.to_csv(os.path.join(OUT_DIR, "results_ledger.csv"), index=False)

tests = paired_tests(led, champion="SelectedStack",
                     baselines=["CSA-SVR", "BestSingle", "FullStack",
                                "AvgEnsemble", "XGB-default", "MLR"])
tests.to_csv(os.path.join(OUT_DIR, "T8_significance_tests.csv"), index=False)
print("T8 — SelectedStack vs baselines (test RMSE, paired over seeds):")
display(tests.round(5))

agg = (led[led.split_kind == "random"]
       .groupby(["model", "metric"])["value"]
       .agg(["mean", "std", "count"]).reset_index())
agg.to_csv(os.path.join(OUT_DIR, "T7_T8_aggregate_metrics.csv"), index=False)
pivot = agg.pivot_table(index="model", columns="metric", values="mean")
keep = [c for c in ["RMSE", "MAE", "R2", "VAF", "NLL", "CRPS",
                    "PICP90", "MPIW90", "PICP95", "MPIW95"] if c in pivot]
print("Aggregate test metrics (mean over seeds):")
display(pivot[keep].round(4))

fig_seed_boxplots(led, ["MLR", "CSA-SVR", "XGB-default", "BestSingle",
                        "AvgEnsemble", "FullStack", "SelectedStack"],
                  OUT_DIR)                                  # F9

## Phase 8 — Interpretability
KernelSHAP on the (model-agnostic) selected stack (**F11**), TreeSHAP
interaction values on the tuned XGBoost exposing the CP × IP coupling
(**F12**), and manual PDP + ICE curves (**F13**). Compare the SHAP ranking
with Zhou et al.'s cosine-amplitude ranking (IP > Z > n > CP > FTC) in the
paper's discussion — agreement validates, disagreement is a talking point.

In [ ]:
def stack_predict_W(Xraw):
    base = np.column_stack([fitted_W[POOL[j]].predict(Xraw)
                            for j in sk_W["cols"]])
    return sk_W["meta"].predict(base)


sv_W, imp_W = shap_on_stack(stack_predict_W, Xtr_W, Xte_W, B,
                            seed_dir(WALKTHROUGH_SEED))        # F11
display(pd.DataFrame({"feature": FEATURES, "mean_abs_SHAP": imp_W})
        .sort_values("mean_abs_SHAP", ascending=False).round(3))
_ = xgb_interactions(fitted_W["xgb"], Xte_W,
                     seed_dir(WALKTHROUGH_SEED))               # F12
pdp_ice(stack_predict_W, Xtr_W, B, seed_dir(WALKTHROUGH_SEED)) # F13

## Phase 9 — Engineering application: cold-region design charts
The tuned NGBoost sweeps the FTC × CP × IP domain (porosity and impedance at
dataset means): mean and **5 %-fractile** DCS contours with the 72
experimental cells overlaid (**F16**), the fractile design table (**T12**)
and the critical freeze–thaw life N_crit at 80 % of FT0 strength (**F16b**).
Validity domain: interpolation inside the tested factorial space of this
saturated Tongchuan sandstone at −20 °C cycling — say so in the caption.

In [ ]:
G_df, ncrit = design_charts(uq_W["ngb"],
                            lambda m, X: _dist_mu_sigma(m, X, Normal),
                            df, cond_stats, B,
                            seed_dir(WALKTHROUGH_SEED))   # F16 / F16b / T12
print("N_crit (cycles to 80% of FT0 strength, 5%-fractile basis):")
display(ncrit[ncrit.basis == "frac5"]
        .pivot_table(index="CP", columns="IP", values="N_crit").round(1))

## Wrap-up
Output inventory and the submission reminders. **Fast-mode numbers are not
paper numbers** — switch `MODE="full"` (seeds 1–10) and rerun top-to-bottom;
caches make partial progress durable. For the manuscript, map outputs to
tables/figures per the methodology document (T1–T12, F2–F16).

In [ ]:
n_png = n_csv = n_json = 0
for root, _, files in os.walk(OUT_DIR):
    for f in files:
        n_png += f.endswith(".png")
        n_csv += f.endswith(".csv")
        n_json += f.endswith(".json")
print(f"outputs under {OUT_DIR}/: {n_png} figures | {n_csv} tables | "
      f"{n_json} json | ledger rows: {len(LEDGER_ROWS)}")
print()
print("Reminders:")
print(" - Paper numbers: MODE='full', SEEDS=1..10 (resumable; run overnight)")
print(" - Cite Meng 2023 (data), Zhou 2024 (benchmark), IOP EES 1337:012072")
print("   2024 (confirm authors), Cai 2025 (Materials) — differentiate each")
print(" - Report the condition-blocked stress split (RUN_BLOCKED=True)")
print(" - State the replicate-sigma train-side caveat (methodology 6.5)")
print("NOTEBOOK RUN COMPLETE")